# Crawl Dữ Liệu Việc Làm CNTT từ VietnamWorks (Mẫu 50 Jobs)

> **URL tìm kiếm**: `https://www.vietnamworks.com/viec-lam?q=it&g=5`
>
> **Kỹ thuật trích xuất**:
> 1. **Kiến trúc nền tảng**: VietnamWorks sử dụng Microservice API backend (`https://ms.vietnamworks.com/job-search/v1.0/search`) cung cấp dữ liệu JSON trực tiếp với đầy đủ trường thông tin (JD, yêu cầu, quyền lợi, kỹ năng, mức lương).
> 2. **Cơ chế bypass**: Dùng `curl_cffi` giả lập Chrome (`impersonate="chrome124"`) và gửi query `jobFunctionId: 5` (Ngành CNTT/Phần mềm) với từ khóa `it`.

In [ ]:
import os
import json
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests
from tqdm import tqdm

## Phần 1: Crawl Dữ Liệu Việc Làm VietnamWorks (List & Details)

API của VietnamWorks trả về cả thông tin tổng quan bài đăng lẫn nội dung JD chi tiết trong cùng một phản hồi.

In [ ]:
def clean_html(raw_html: str) -> str:
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator="\n", strip=True)

def crawl_vietnamworks_jobs(target_count: int = 50) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Crawl danh sách và chi tiết 50 việc làm IT mẫu từ VietnamWorks API.
    """
    session = requests.Session(impersonate="chrome124")
    headers = {
        'Referer': 'https://www.vietnamworks.com/',
        'Origin': 'https://www.vietnamworks.com',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Content-Type': 'application/json'
    }
    payload = {
        "query": "it",
        "jobFunctionId": [{"parentId": 5, "childrenIds": [-1]}],
        "page": 0,
        "hitsPerPage": target_count
    }
    
    print(f"[*] Gửi request tìm kiếm IT từ VietnamWorks (Mục tiêu: {target_count} jobs)...")
    resp = session.post("https://ms.vietnamworks.com/job-search/v1.0/search", json=payload, headers=headers, timeout=20)
    if resp.status_code != 200:
        raise RuntimeError(f"VNW API error: {resp.status_code} - {resp.text[:200]}")
        
    data = resp.json().get("data", [])
    print(f"[+] Nhận thành công {len(data)} công việc từ VietnamWorks.")
    
    list_records = []
    detail_records = []
    
    for item in tqdm(data[:target_count], desc="Xử lý dữ liệu VNW"):
        job_id = str(item.get("jobId", ""))
        detail_id = f"vietnamworks_{job_id}"
        job_url = item.get("jobUrl", "")
        title = item.get("jobTitle", "")
        company_name = item.get("companyName", "")
        company_logo = item.get("companyLogo", "")
        company_url = item.get("companyUrl", "")
        
        pretty_sal = item.get("prettySalary") or item.get("salary") or "Thoả thuận"
        
        locations = item.get("locations") or []
        loc_str = ", ".join([l.get("cityName", "") for l in locations if isinstance(l, dict) and l.get("cityName")]) if isinstance(locations, list) else str(locations)
        if not loc_str and item.get("address"):
            loc_str = item.get("address")
            
        exp_years = item.get("yearsOfExperience")
        exp_str = f"{exp_years} năm" if exp_years else "Không yêu cầu / Không xác định"
        
        posted_at = item.get("approvedOn", "") or item.get("createdOn", "")
        updated_at = item.get("lastUpdatedOn", "")
        deadline = item.get("expiredOn", "")
        crawled_at = datetime.now().isoformat()
        
        # 1. List Schema
        list_records.append({
            "source": "vietnamworks",
            "source_job_id": job_id,
            "job_url": job_url,
            "company_logo_url": company_logo,
            "tile_raw": title,
            "company_name_raw": company_name,
            "location_raw": loc_str,
            "salary_raw": str(pretty_sal),
            "experience_raw": exp_str,
            "posted_at_raw": str(posted_at),
            "crawled_at": crawled_at
        })
        
        # 2. Detail Schema
        skills_list = item.get("skills") or []
        skills_raw = ", ".join([s.get("skillName", "") for s in skills_list if isinstance(s, dict) and s.get("skillName")])
        
        benefits_list = []
        for b in (item.get("benefits") or []):
            if isinstance(b, dict):
                b_name = b.get("benefitNameVI") or b.get("benefitName") or ""
                b_val = b.get("benefitValue") or ""
                benefits_list.append(f"{b_name}: {b_val}" if b_val else b_name)
        benefits_raw = "\n".join([b for b in benefits_list if b])
        
        work_type = item.get("typeWorking", {}).get("name") if isinstance(item.get("typeWorking"), dict) else ""
        emp_type = item.get("jobLevelVI") or item.get("jobLevel", "")
        comp_size = item.get("companySizeVI") or item.get("companySize", "")
        
        ind_list = item.get("industries") or []
        industries = ", ".join([ind.get("industryNameVI", "") or ind.get("industryName", "") for ind in ind_list if isinstance(ind, dict)])
        
        desc_text = clean_html(item.get("jobDescription", ""))
        req_text = clean_html(item.get("jobRequirement", ""))
        
        detail_records.append({
            "detail_id": detail_id,
            "source": "vietnamworks",
            "source_job_id": job_id,
            "job_url": job_url,
            "title_raw": title,
            "company_name_raw": company_name,
            "company_url": company_url,
            "company_size_raw": comp_size,
            "company_location_raw": loc_str,
            "industry_raw": industries,
            "salary_raw": str(pretty_sal),
            "experience_raw": exp_str,
            "education_raw": "",
            "location_raw": loc_str,
            "work_mode_raw": work_type,
            "employment_type_raw": emp_type,
            "working_time_raw": "",
            "overview_raw": item.get("summary", "") or "",
            "responsibilities_raw": desc_text,
            "requirements_raw": req_text,
            "benefits_raw": benefits_raw,
            "required_skills_raw": skills_raw,
            "preferred_skills_raw": "",
            "published_at_raw": str(posted_at),
            "updated_at_raw": str(updated_at),
            "deadline_raw": str(deadline),
            "crawled_at": crawled_at
        })
        
    return pd.DataFrame(list_records), pd.DataFrame(detail_records)

In [ ]:
df_vnw_list, df_vnw_detail = crawl_vietnamworks_jobs(target_count=50)

# Lưu dữ liệu vào data/raw
output_dir = os.path.join("..", "data", "raw")
os.makedirs(output_dir, exist_ok=True)

list_csv = os.path.join(output_dir, "vietnamworks_jobs_raw.csv")
list_json = os.path.join(output_dir, "vietnamworks_jobs_raw.json")
detail_csv = os.path.join(output_dir, "vietnamworks_job_details_raw.csv")
detail_json = os.path.join(output_dir, "vietnamworks_job_details_raw.json")

df_vnw_list.to_csv(list_csv, index=False, encoding="utf-8-sig")
df_vnw_list.to_json(list_json, orient="records", force_ascii=False, indent=2)

df_vnw_detail.to_csv(detail_csv, index=False, encoding="utf-8-sig")
df_vnw_detail.to_json(detail_json, orient="records", force_ascii=False, indent=2)

print(f"[+] Đã lưu {len(df_vnw_detail)} jobs vào:")
print(f"    - {list_csv}")
print(f"    - {detail_csv}")

print("Shape Detail:", df_vnw_detail.shape)
df_vnw_detail.head(3)